In [ ]:
from google.colab import files
import pandas as pd

print("Upload clean_master_tract@1.csv")
uploaded = files.upload()

Upload clean_master_tract@1.csv (or whatever you named it)


Saving clean_master_tract@1.csv to clean_master_tract@1.csv


In [ ]:
# Load the cleaned master file
df = pd.read_csv("clean_master_tract@1.csv")

# Clean tract label: "Tract 1.02;PhiladelphiaCounty;Pennsylvania" → "Tract 1.02"
df["tract_label"] = df["tract.y"].str.split(";").str[0].str.strip()

# Clean tract ID: drop the trailing .0 from the GEOID
df["tract_id"] = df["tract"].astype(int).astype(str)

# Compute majority race (>50% threshold) — matches hexmap_2.ipynb logic
def majority_race(row):
    races = {
        "White":    row["pct_white"],
        "Black":    row["pct_black"],
        "Asian":    row["pct_asian"],
        "Hispanic": row["pct_hispanic"],
    }
    if pd.isna(row["pct_white"]):
        return None
    top = max(races, key=races.get)
    return top if races[top] > 50 else "None"

df["majority_race"] = df.apply(majority_race, axis=1)

# Drop columns we don't need
df = df.drop(columns=["tract", "tract.y", "COD", "PRD"], errors="ignore")

print(f"Loaded {len(df)} rows across years {sorted(df['year'].unique())}")
print(f"Majority race distribution:")
print(df['majority_race'].value_counts(dropna=False))

Loaded 411 rows across years [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Majority race distribution:
majority_race
Black       163
White       162
None         68
Hispanic     18
Name: count, dtype: int64


In [ ]:
# Filter to the two years we want to compare
comp = df[df["year"].isin([2022, 2023])].copy()

# Keep only tracts that appear in BOTH years
tracts_both = comp.groupby("tract_id")["year"].nunique().eq(2)
keep_ids = tracts_both[tracts_both].index
comp = comp[comp["tract_id"].isin(keep_ids)]

# Compare each tract's majority_race across the two years
race_by_year = comp.pivot_table(
    index="tract_id",
    columns="year",
    values="majority_race",
    aggfunc="first"
)
race_by_year.columns = [f"majority_race_{y}" for y in race_by_year.columns]

# Flag tracts where majority_race differs between 2022 and 2023
race_by_year["race_changed"] = (
    race_by_year["majority_race_2022"] != race_by_year["majority_race_2023"]
)

# Report any mismatches before continuing
mismatches = race_by_year[race_by_year["race_changed"]]
if len(mismatches) > 0:
    print(f"⚠️  {len(mismatches)} tract(s) have different majority_race in 2022 vs 2023:")
    print(mismatches)
    print()
else:
    print("✓ All tracts have consistent majority_race across 2022 and 2023\n")

# Pivot the numeric columns: each tract gets a column for each year
comp_wide = comp.pivot_table(
    index=["tract_id", "tract_label"],
    columns="year",
    values=["median_ratio", "median_sale_price", "median_assessed_value", "n_sales"]
).reset_index()

# Flatten multi-index column names: ('median_ratio', 2022) -> 'median_ratio_2022'
comp_wide.columns = [
    f"{a}_{b}" if b != "" else a for a, b in comp_wide.columns
]

# Merge in the race columns and the change flag
comp_wide = comp_wide.merge(
    race_by_year.reset_index(),
    on="tract_id",
    how="left"
)

# Canonical majority_race = the 2023 value (matches existing charts)
comp_wide["majority_race"] = comp_wide["majority_race_2023"]

comp_wide.to_csv("comparison_2022_2023.csv", index=False)
print(f"comparison_2022_2023.csv: {len(comp_wide)} rows")
print("\nFirst few rows:")
print(comp_wide.head())
print("\nColumns:")
print(comp_wide.columns.tolist())

⚠️  1 tract(s) have different majority_race in 2022 vs 2023:
            majority_race_2022 majority_race_2023  race_changed
tract_id                                                       
42101025700              White               None          True

comparison_2022_2023.csv: 49 rows

First few rows:
      tract_id tract_label  median_assessed_value_2022  \
0  42101000102  Tract 1.02                    900000.0   
1  42101000300     Tract 3                   1079450.0   
2  42101000500     Tract 5                    696400.0   
3  42101000806  Tract 8.06                   1547100.0   
4  42101002400    Tract 24                    360400.0   

   median_assessed_value_2023  median_ratio_2022  median_ratio_2023  \
0                   1032100.0           0.842583           1.077379   
1                   1176850.0           0.894053           1.133070   
2                    773200.0           1.045757           0.994746   
3                   1707250.0           0.956192           0.9

In [ ]:
files.download("comparison_2022_2023.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>